## Combine & Status
Scans `RESULTS/` across all sizes and trials, reports failures/missing, and writes one combined CSV per trial to `COMBINED/`.

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict


In [2]:
RESULTS_DIR  = Path("RESULTS")
COMBINED_DIR = Path("COMBINED")
OUTPUT_DIR   = Path("OUTPUT")          # to know what was expected
IFT_SUBPATH  = "CSV/InterfacialProperties/feed_1_interfacial_results.csv"


### Discover expected sizes and trials from OUTPUT/

In [3]:
# What was expected to run (from INPUT csv files)
expected = {}   # (size, trial) -> n_compositions
if OUTPUT_DIR.exists():
    for size_dir in sorted(OUTPUT_DIR.iterdir()):
        if not size_dir.is_dir():
            continue
        for csv in sorted(size_dir.glob("trial_*.csv")):
            trial = csv.stem
            n = sum(1 for _ in csv.open()) - 1   # subtract header
            expected[(size_dir.name, trial)] = n

print(f"Expected: {len(expected)} trials across {len({s for s,_ in expected})} sizes")
for size in sorted({s for s,_ in expected}):
    trials = sorted(t for s,t in expected if s == size)
    print(f"  {size}: {len(trials)} trials, {expected[(size, trials[0])]} compositions/trial")


Expected: 80 trials across 4 sizes
  N025: 20 trials, 25 compositions/trial
  N050: 20 trials, 50 compositions/trial
  N075: 20 trials, 75 compositions/trial
  N100: 20 trials, 100 compositions/trial


### Scan RESULTS/ and classify tasks

In [4]:
failures  = {}   # (size, trial) -> [task_ids with no output]
missing   = {}   # (size, trial) -> [task_ids with no folder at all]
successes = {}   # (size, trial) -> {task_id: best_folder}

for (size, trial), n_expected in sorted(expected.items()):
    trial_dir = RESULTS_DIR / size / trial

    if not trial_dir.exists():
        # Entire trial has no results yet
        missing[(size, trial)] = list(range(n_expected))
        continue

    # Group result folders by task_id
    by_task = defaultdict(list)
    for folder in trial_dir.iterdir():
        if not folder.is_dir():
            continue
        parts = folder.name.rsplit("_", 1)
        if len(parts) == 2 and parts[1].isdigit():
            by_task[int(parts[1])].append(folder)

    trial_failures = []
    trial_missing  = []
    trial_successes = {}

    for task_id in range(n_expected):
        folders = by_task.get(task_id, [])
        if not folders:
            trial_missing.append(task_id)
        else:
            ok = [f for f in folders if (f / IFT_SUBPATH).exists()]
            if not ok:
                trial_failures.append(task_id)
            else:
                # Pick highest job ID if multiple successful runs
                best = sorted(ok, key=lambda f: int(f.name.rsplit("_", 1)[0]))[-1]
                trial_successes[task_id] = best

    if trial_failures:
        failures[(size, trial)] = trial_failures
    if trial_missing:
        missing[(size, trial)] = trial_missing
    if trial_successes:
        successes[(size, trial)] = trial_successes

n_success = sum(len(v) for v in successes.values())
n_fail    = sum(len(v) for v in failures.values())
n_miss    = sum(len(v) for v in missing.values())
print(f"Successful tasks : {n_success}")
print(f"Failed tasks     : {n_fail}   (ran but produced no output)")
print(f"Missing tasks    : {n_miss}  (never ran)")


Successful tasks : 5000
Failed tasks     : 0   (ran but produced no output)
Missing tasks    : 0  (never ran)


### What is missing / failed

In [5]:
if not failures and not missing:
    print("All tasks completed successfully — nothing missing.")
else:
    if missing:
        print("=== Never ran (no result folder) ===")
        for (size, trial), task_ids in sorted(missing.items()):
            if len(task_ids) == expected[(size, trial)]:
                print(f"  {size}/{trial}  →  entire trial not submitted yet")
            else:
                print(f"  {size}/{trial}  →  {len(task_ids)} tasks never ran: {task_ids}")

    if failures:
        print("\n=== Ran but failed (no CSV output) ===")
        for (size, trial), task_ids in sorted(failures.items()):
            print(f"  {size}/{trial}  →  {len(task_ids)} failed tasks: {task_ids}")


All tasks completed successfully — nothing missing.


### Combine successful results per trial

In [6]:
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

n_written = 0

for (size, trial), task_map in sorted(successes.items()):
    out_dir = COMBINED_DIR / size
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{trial}.csv"

    dfs = []
    for task_id, folder in sorted(task_map.items()):
        try:
            df = pd.read_csv(folder / IFT_SUBPATH)
            df.insert(0, "task_id", task_id)
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: {folder.name}: {e}")

    if not dfs:
        continue

    pd.concat(dfs, ignore_index=True).to_csv(out_path, index=False)
    n_written += 1

print(f"Written {n_written} combined CSVs to {COMBINED_DIR}/")


Written 80 combined CSVs to COMBINED/


### Summary per size

In [7]:
print(f"{'Size':<8} {'Trials done':>12} {'Tasks OK':>10} {'Failed':>8} {'Missing':>9}")
print("-" * 52)

for size in sorted({s for s,_ in expected}):
    trials_all     = [(size, t) for s,t in expected if s == size]
    trials_done    = sum(1 for k in trials_all if k in successes)
    tasks_ok       = sum(len(v) for k,v in successes.items() if k[0] == size)
    tasks_failed   = sum(len(v) for k,v in failures.items()  if k[0] == size)
    tasks_missing  = sum(len(v) for k,v in missing.items()   if k[0] == size)
    print(f"{size:<8} {trials_done:>7}/{len(trials_all):<4} {tasks_ok:>10} {tasks_failed:>8} {tasks_missing:>9}")


Size      Trials done   Tasks OK   Failed   Missing
----------------------------------------------------
N025          20/20          500        0         0
N050          20/20         1000        0         0
N075          20/20         1500        0         0
N100          20/20         2000        0         0
